# Multiaxial Universal ML plasticity model

To generalise the formulation in [VonMisesMetal1D](https://github.com/sinaplatform/plasticity-models/blob/main/VonMisesMetal1D.ipynb) into a multiaxial case, first consider the elastic case as follow

$$ d\sigma= \mathbb{C}^e d\varepsilon^e$$

where in Matrix Form (Voigt Notation) $\mathbb{C}^e$ is 6 by 6 full matrix, it means that the stress increment in one direction is coupled with all loading directions.

$\textbf{IMPORTANT obeservation:}$ Accourding to the 1D formulation in [VonMisesMetal1D](https://github.com/sinaplatform/plasticity-models/blob/main/VonMisesMetal1D.ipynb), it seems that $\mathbb{C}$ itself if coupled with intenal state variables is able to handle history dependency!


Let us forget about the seperation of elastic and plastic part of material response and assume that the material is nonlinear under any loading magnitude. 

$$ d\sigma= \mathbb{C} d\varepsilon$$

Note that $\sigma$ and $\varepsilon$ are second order tensors in 3D and the material stiffness (tangental or Jacobian) $\mathbb{C}$ is a fourth-order tensor

$$
d \sigma_{ij} =  \mathbb{C}_{ijkl} d \varepsilon_{kl}
$$

where 

$$ 
\mathbb{C}_{ijkl} = \frac{\partial d \sigma_{ij}}{\partial d \varepsilon_{kl}}.
$$

In Matrix Form (Voigt Notation):

$$
d \sigma = 
\begin{bmatrix}
d \sigma_{11} \\
d \sigma_{22} \\
d \sigma_{33} \\
d \sigma_{23} \\
d \sigma_{31} \\
d \sigma_{12}
\end{bmatrix}
,~~~
d \varepsilon = 
\begin{bmatrix}
d \varepsilon_{11} \\
d \varepsilon_{22} \\
d \varepsilon_{33} \\
2 d \varepsilon_{23} \\
2 d \varepsilon_{31} \\
2 d \varepsilon_{12}
\end{bmatrix}
$$

These are different in ABAUQS.

$$
\mathbb{C} =
\begin{bmatrix}
C_{11} & C_{12} & C_{13} & C_{14} & C_{15} & C_{16} \\
C_{21} & C_{22} & C_{23} & C_{24} & C_{25} & C_{26} \\
C_{31} & C_{32} & C_{33} & C_{34} & C_{35} & C_{36} \\
C_{41} & C_{42} & C_{43} & C_{44} & C_{45} & C_{46} \\
C_{51} & C_{52} & C_{53} & C_{54} & C_{55} & C_{56} \\
C_{61} & C_{62} & C_{63} & C_{64} & C_{65} & C_{66} \\
\end{bmatrix}.
$$


<!-- Therefore a universal dynamic ML material plasticity model will look like this:

$$ \left\{
    \begin{aligned}
        d\sigma &= f(h) d\varepsilon\\
        dh &=g(h) d\varepsilon \\
    \end{aligned}
    \right. 
$$

or 

$$ \left\{
    \begin{aligned}
        \frac{d\sigma}{d\varepsilon}  &= f(h) \\
        \frac{dh}{d\varepsilon} &=g(h) \\
    \end{aligned}
    \right. 
$$

where $f$ and $g$ are neural operators (NO) with dimensions $f: \mathbb{R}^k \rightarrow \mathbb{R}^6 $ and $g: \mathbb{R}^k \rightarrow \mathbb{R}^6 $ for a 3D case. Here, $k$ is the number of states and we consider the Jacobian of states as below which for training purposes flattens.

$$
\frac{d\boldsymbol{h}}{d\boldsymbol{\varepsilon}} =
\begin{bmatrix}
\frac{\partial h^1}{\partial \varepsilon^1} & \frac{\partial h^1}{\partial \varepsilon^2} & \cdots \\
\frac{\partial h^2}{\partial \varepsilon^1} & \frac{\partial h^2}{\partial \varepsilon^2} & \cdots \\
\vdots & \vdots & \ddots
\end{bmatrix}
$$

Yet, we can use only one NN for all state evolutions or one NN for each state variable in all directions.  -->

## New state space formulation
Considering the [combined isotropic/kinematic Von Mises model](https://github.com/sinaplatform/plasticity-models/blob/main/VonMisesMetal1D.ipynb)

$$ \left\{
    \begin{aligned}
        \frac{d\sigma}{d\varepsilon}  &= f(\sigma,h) \\
        \frac{dh}{d\varepsilon} &=g(\sigma,h) \\
    \end{aligned}
    \right. 
$$

if $ \boldsymbol{y} = \begin{bmatrix}
\sigma \\
h
\end{bmatrix} $ the universal state space ML model will be as follows:

$$ \left\{
    \begin{aligned}
        d\boldsymbol{y}(s) &= \mathscr{NN}(\boldsymbol{y}(s), \theta_{w,b}) ~ d \boldsymbol{\varepsilon}(s) \\
        y(0) & = g(\begin{bmatrix}
                    \sigma (0) \\
                    h (0)
                    \end{bmatrix}) \\
        \hat{\sigma} &= \sigma(1)          
    \end{aligned}
    \right. \quad s \in[0,1]
$$


where $\mathscr{NN}$ is neural operators (NO) with dimensions $\mathscr{NN}: \mathbb{R}^{6+k} \rightarrow \mathbb{R}^{6+k \times 6} $ for a 3D case. 
Here, $s$ is the increment step, and  $k$ is the number of states.
Note that the first 6 raws in the output is a symmetric matrix and therefore the learnable output dimension is only $21 + 6 \times k$.
This architecture directly learns material jacobian and jacobian of internal state variables ($h^k$).

$$
\frac{d\boldsymbol{y}}{d\boldsymbol{\varepsilon}} =
\begin{bmatrix}
\frac{\partial \sigma^1}{\partial \varepsilon^1} & \frac{\partial \sigma^1}{\partial \varepsilon^2} & \frac{\partial \sigma^1}{\partial \varepsilon^3} &
\frac{\partial \sigma^1}{\partial \varepsilon^4} &  \frac{\partial \sigma^1}{\partial \varepsilon^5} & \frac{\partial \sigma^1}{\partial \varepsilon^6} 
 \\
        & \frac{\partial \sigma^2}{\partial \varepsilon^2} & \frac{\partial \sigma^2}{\partial \varepsilon^3} &
\frac{\partial \sigma^2}{\partial \varepsilon^4} &  \frac{\partial \sigma^2}{\partial \varepsilon^5} & \frac{\partial \sigma^2}{\partial \varepsilon^6} 
 \\
     &   & \frac{\partial \sigma^3}{\partial \varepsilon^3} &
\frac{\partial \sigma^3}{\partial \varepsilon^4} &  \frac{\partial \sigma^3}{\partial \varepsilon^5} & \frac{\partial \sigma^3}{\partial \varepsilon^6} 
 \\
     &   &   &
\frac{\partial \sigma^4}{\partial \varepsilon^4} &  \frac{\partial \sigma^4}{\partial \varepsilon^5} & \frac{\partial \sigma^4}{\partial \varepsilon^6} 
 \\
     & \text{sym}  &   &
     &  \frac{\partial \sigma^5}{\partial \varepsilon^5} & \frac{\partial \sigma^5}{\partial \varepsilon^6} 
 \\
     &   &   &
     &       & \frac{\partial \sigma^6}{\partial \varepsilon^6} 
 \\ \hdashline

\frac{\partial h^1}{\partial \varepsilon^1} & \frac{\partial h^1}{\partial \varepsilon^2} & \frac{\partial h^1}{\partial \varepsilon^3} &
\frac{\partial h^1}{\partial \varepsilon^4} &  \frac{\partial h^1}{\partial \varepsilon^5} & \frac{\partial h^1}{\partial \varepsilon^6} 
 \\
\frac{\partial h^2}{\partial \varepsilon^1} & \frac{\partial h^2}{\partial \varepsilon^2} & \frac{\partial h^1}{\partial \varepsilon^3} &
\frac{\partial h^2}{\partial \varepsilon^4} &  \frac{\partial h^2}{\partial \varepsilon^5} & \frac{\partial h^2}{\partial \varepsilon^6}  \\
\vdots & \vdots & \vdots &\vdots &\vdots &\vdots 
\end{bmatrix}
$$


In [3]:
import os
import argparse
import time
from datetime import datetime
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

import torchdyn
from torchdyn.core import NeuralODE
from typing import Tuple

from torchdiffeq import odeint

import matplotlib.pyplot as plt
from ipywidgets import interact
%matplotlib qt
import pyqtgraph as pg
from pyqtgraph.Qt import QtGui
plt.rcdefaults()

# figure styling
font=16
font_axis=20
plt.rcParams.update({'font.size': font})  # Set the desired font size
# plt.rcParams['text.usetex'] = True
plt.rcParams['figure.facecolor'] = 'white'  # Background color for figures
plt.rcParams['axes.facecolor'] = 'white'    # Background color for axes

# Define the neural network model training parameters
args = argparse.Namespace(
      method='dopri5',
      data_size=200,
      batch_time=10,
      batch_size=20,
      niters=2000,
      test_freq=20,
      viz=True,
      gpu=0,
      adjoint=False
  )

if args.adjoint:
    from torchdiffeq import odeint_adjoint as odeint
else:
    from torchdiffeq import odeint

device = torch.device('cuda:' + str(args.gpu) if torch.cuda.is_available() else 'cpu')

In [4]:
# Record the start time
start_time = datetime.now()
print(f"Run started at: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

# import and navigate through the dataset 
time = np.expand_dims(np.load('t.npy'), axis=-1)
strain_full = np.expand_dims(np.load('strain.npy'), axis=-1)
stress_full = np.expand_dims(np.load('S.npy'), axis=-1)
state = np.expand_dims(np.load('state.npy'), axis=-1) # z in the paper

# Record the end time
end_time = datetime.now()
print(f"Run ended at: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")

# Optionally, print the duration
duration = end_time - start_time
print(f"Total run duration: {duration}")

Run started at: 2024-12-12 11:17:18
Run ended at: 2024-12-12 11:17:18
Total run duration: 0:00:00.015097


In [ ]:
def get_batch():
    s = torch.from_numpy(np.random.choice(np.arange(args.data_size - args.batch_time, dtype=np.int64), args.batch_size, replace=False))
    batch_y0 = true_y[s]  # (M, D)
    batch_t = t[:args.batch_time]  # (T)
    batch_y = torch.stack([true_y[s + i] for i in range(args.batch_time)], dim=0)  # (T, M, D)
    return batch_y0.to(device), batch_t.to(device), batch_y.to(device)


def makedirs(dirname):
    if not os.path.exists(dirname):
        os.makedirs(dirname)


if args.viz:
    makedirs('png')
    fig = plt.figure(figsize=(12, 4), facecolor='white')
    ax_traj = fig.add_subplot(131, frameon=False)
    ax_phase = fig.add_subplot(132, frameon=False)
    ax_vecfield = fig.add_subplot(133, frameon=False)
    plt.show(block=False)


def visualize(true_y, pred_y, odefunc, itr):

    if args.viz:

        ax_traj.cla()
        ax_traj.set_title('Trajectories')
        ax_traj.set_xlabel('t')
        ax_traj.set_ylabel('x,y')
        ax_traj.plot(t.cpu().numpy(), true_y.cpu().numpy()[:, 0, 0], t.cpu().numpy(), true_y.cpu().numpy()[:, 0, 1], 'g-')
        ax_traj.plot(t.cpu().numpy(), pred_y.cpu().numpy()[:, 0, 0], '--', t.cpu().numpy(), pred_y.cpu().numpy()[:, 0, 1], 'b--')
        ax_traj.set_xlim(t.cpu().min(), t.cpu().max())
        ax_traj.set_ylim(-2, 2)
        ax_traj.legend()

        ax_phase.cla()
        ax_phase.set_title('Phase Portrait')
        ax_phase.set_xlabel('x')
        ax_phase.set_ylabel('y')
        ax_phase.plot(true_y.cpu().numpy()[:, 0, 0], true_y.cpu().numpy()[:, 0, 1], 'g-')
        ax_phase.plot(pred_y.cpu().numpy()[:, 0, 0], pred_y.cpu().numpy()[:, 0, 1], 'b--')
        ax_phase.set_xlim(-2, 2)
        ax_phase.set_ylim(-2, 2)

        ax_vecfield.cla()
        ax_vecfield.set_title('Learned Vector Field')
        ax_vecfield.set_xlabel('x')
        ax_vecfield.set_ylabel('y')

        y, x = np.mgrid[-2:2:21j, -2:2:21j]
        dydt = odefunc(0, torch.Tensor(np.stack([x, y], -1).reshape(21 * 21, 2)).to(device)).cpu().detach().numpy()
        mag = np.sqrt(dydt[:, 0]**2 + dydt[:, 1]**2).reshape(-1, 1)
        dydt = (dydt / mag)
        dydt = dydt.reshape(21, 21, 2)

        ax_vecfield.streamplot(x, y, dydt[:, :, 0], dydt[:, :, 1], color="black")
        ax_vecfield.set_xlim(-2, 2)
        ax_vecfield.set_ylim(-2, 2)

        fig.tight_layout()
        plt.savefig('png/{:03d}'.format(itr))
        plt.draw()
        plt.pause(0.1)


class ODEFunc(nn.Module):

    def __init__(self):
        super(ODEFunc, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(2, 50),
            nn.Tanh(),
            nn.Linear(50, 2),
        )

        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0, std=0.1)
                nn.init.constant_(m.bias, val=0)

    def forward(self, t, y):
        return self.net(y)


class RunningAverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self, momentum=0.99):
        self.momentum = momentum
        self.reset()

    def reset(self):
        self.val = None
        self.avg = 0

    def update(self, val):
        if self.val is None:
            self.avg = val
        else:
            self.avg = self.avg * self.momentum + val * (1 - self.momentum)
        self.val = val


if __name__ == '__main__':

    ii = 0

    func = ODEFunc().to(device)
    
    optimizer = optim.RMSprop(func.parameters(), lr=1e-3)
    end = time.time()

    time_meter = RunningAverageMeter(0.97)
    
    loss_meter = RunningAverageMeter(0.97)

    for itr in range(1, args.niters + 1):
        optimizer.zero_grad()
        batch_y0, batch_t, batch_y = get_batch()
        pred_y = odeint(func, batch_y0, batch_t).to(device)
        loss = torch.mean(torch.abs(pred_y - batch_y))
        loss.backward()
        optimizer.step()

        time_meter.update(time.time() - end)
        loss_meter.update(loss.item())

        if itr % args.test_freq == 0:
            with torch.no_grad():
                pred_y = odeint(func, true_y0, t)
                loss = torch.mean(torch.abs(pred_y - true_y))
                print('Iter {:04d} | Total Loss {:.6f}'.format(itr, loss.item()))
                visualize(true_y, pred_y, func, ii)
                ii += 1

        end = time.time()

In [ ]:
# Record the start time
start_time = datetime.now()
print(f"Run started at: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

# import and navigate through the dataset
time = np.load('time.npy')
output = np.load('ave_tensor.npy')

strain_full = np.zeros((output.shape[0], output.shape[1], 6))
stress_full = np.zeros((output.shape[0], output.shape[1], 6))

# Fill the strain tensor components
strain_full[:, :, 0] = output[:, :, 0]  # E11
strain_full[:, :, 1] = output[:, :, 1]  # E22
strain_full[:, :, 2] = output[:, :, 2]  # E33
strain_full[:, :, 3] = output[:, :, 3]  # E12 
strain_full[:, :, 4] = output[:, :, 4]  # E13
strain_full[:, :, 5] = output[:, :, 5]  # E23

# Fill the stress tensor components
stress_full[:, :, 0] = output[:, :, 6]   # S11
stress_full[:, :, 1] = output[:, :, 7]   # S22
stress_full[:, :, 2] = output[:, :, 8]   # S33
stress_full[:, :, 3] = output[:, :, 9]   # S12 
stress_full[:, :, 4] = output[:, :, 10]  # S13
stress_full[:, :, 5] = output[:, :, 11]  # S23

# Record the end time
end_time = datetime.now()
print(f"Run ended at: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")

# Optionally, print the duration
duration = end_time - start_time
print(f"Total run duration: {duration}")

In [ ]:

class SymmetricMatrix(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        self.n_elements = (dim * (dim + 1)) // 2
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.shape[0]
        output = torch.zeros(batch_size, self.dim, self.dim, device=x.device)
        
        idx = 0
        for i in range(self.dim):
            for j in range(i, self.dim):
                output[:, i, j] = x[:, idx]
                output[:, j, i] = x[:, idx]
                idx += 1
        
        return output

class ConstitutiveODEFunc(nn.Module):
    def __init__(self, k_states: int = 3, hidden_dim: int = 50):
        super().__init__()
        self.k_states = k_states
        
        # Input dimension: 6 (stress) + k (internal variables)
        input_dim = 6 + k_states
        
        # Output dimension: 21 (symmetric 6x6 matrix) + 6*k (internal variables jacobian)
        output_dim = 21 + 6 * k_states
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )
        
        self.symmetric_layer = SymmetricMatrix(6)
        
        # Initialize weights
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0, std=0.1)
                nn.init.constant_(m.bias, val=0)
    
    def forward(self, t, y):
        # Split input into stress and internal variables
        stress = y[..., :6]
        h = y[..., 6:]
        
        # Get network output
        network_output = self.net(y)
        
        # Split output into stress and internal variables derivatives
        stress_deriv = network_output[..., :21]
        h_deriv = network_output[..., 21:]
        
        # Convert stress derivatives to symmetric matrix and flatten
        stress_matrix = self.symmetric_layer(stress_deriv)
        stress_matrix_flat = stress_matrix.reshape(stress_matrix.shape[0], -1)
        
        # Concatenate with internal variables derivatives
        return torch.cat([stress_matrix_flat[..., :6], h_deriv], dim=-1)

class RunningAverageMeter(object):
    def __init__(self, momentum=0.99):
        self.momentum = momentum
        self.reset()

    def reset(self):
        self.val = None
        self.avg = 0

    def update(self, val):
        if self.val is None:
            self.avg = val
        else:
            self.avg = self.avg * self.momentum + val * (1 - self.momentum)
        self.val = val

def visualize(true_y, pred_y, t, itr):
    """Visualize the first component of stress and internal variables"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Plot stress components
    ax1.set_title('Stress Components')
    ax1.plot(t.cpu().numpy(), true_y.cpu().numpy()[:, 0, :6], 'g-', alpha=0.3, label='True')
    ax1.plot(t.cpu().numpy(), pred_y.cpu().numpy()[:, 0, :6], 'b--', alpha=0.3, label='Predicted')
    ax1.set_xlabel('t')
    ax1.set_ylabel('Stress')
    ax1.legend()
    
    # Plot internal variables
    ax2.set_title('Internal Variables')
    ax2.plot(t.cpu().numpy(), true_y.cpu().numpy()[:, 0, 6:], 'g-', alpha=0.3, label='True')
    ax2.plot(t.cpu().numpy(), pred_y.cpu().numpy()[:, 0, 6:], 'b--', alpha=0.3, label='Predicted')
    ax2.set_xlabel('t')
    ax2.set_ylabel('Internal Variables')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig(f'constitutive_model_{itr:03d}.png')
    plt.close()

def train_model(true_y0, t, true_y, k_states=3, niters=2000, batch_time=10, batch_size=20, test_freq=20, device='cuda'):
    func = ConstitutiveODEFunc(k_states=k_states).to(device)
    optimizer = optim.RMSprop(func.parameters(), lr=1e-3)
    
    time_meter = RunningAverageMeter(0.97)
    loss_meter = RunningAverageMeter(0.97)
    
    batch_y0 = true_y0.to(device)
    batch_t = t[:batch_time].to(device)
    
    for itr in range(1, niters + 1):
        optimizer.zero_grad()
        
        # Forward pass
        pred_y = odeint(func, batch_y0, batch_t).to(device)
        
        # Compute loss
        loss = torch.mean(torch.abs(pred_y - true_y[:batch_time]))
        loss.backward()
        optimizer.step()
        
        time_meter.update(time.time() - end)
        loss_meter.update(loss.item())
        
        if itr % test_freq == 0:
            with torch.no_grad():
                pred_y = odeint(func, true_y0, t)
                loss = torch.mean(torch.abs(pred_y - true_y))
                print(f'Iter {itr:04d} | Total Loss {loss.item():.6f}')
                visualize(true_y, pred_y, t, itr // test_freq)
        
        end = time.time()
    
    return func

# Example usage:
if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Generate dummy data for demonstration
    data_size = 200
    k_states = 3
    total_dim = 6 + k_states  # 6 stress components + k internal variables
    
    # Create dummy initial conditions and true trajectory
    true_y0 = torch.randn(1, total_dim).to(device)
    t = torch.linspace(0., 25., data_size).to(device)
    
    # Generate dummy true trajectory (replace this with your actual data)
    true_y = torch.stack([true_y0 * torch.exp(-0.1 * t_) + torch.randn_like(true_y0) * 0.1 
                         for t_ in t], dim=0)
    
    # Train the model
    model = train_model(true_y0, t, true_y, k_states=k_states)